In [1]:
import json
import pickle
from pathlib import Path

import numpy as np
import pandas as pd

from measurements import compute_measurements

ANNOTATION_PATH = Path("../../../data/Foot13/annotations/foot13_test.json")
PREDICTION_PATH = "../../test_results/{model}_predictions.pkl"
OUTPUT_DIR = Path("../../test_results/0_measurements")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


In [2]:
with ANNOTATION_PATH.open(encoding="utf-8-sig") as f:
    coco = json.load(f)

reference_points = {
    ann["image_id"]: np.array(ann["keypoints"], dtype=float).reshape(13, 3)[:, :2]
    for ann in coco["annotations"]
}
image_paths = {image["id"]: image["file_name"] for image in coco["images"]}
sample_names = {image_id: Path(file_name).stem for image_id, file_name in image_paths.items()}


def make_table(keypoints):
    rows = []
    for image_id in sorted(reference_points):
        rows.append({
            "sample_name": sample_names[image_id],
            "image_id": image_id,
            "image_path": image_paths[image_id],
            **compute_measurements(keypoints[image_id]),
        })
    return pd.DataFrame(rows)


In [3]:
reference_table = make_table(reference_points)
reference_table.to_csv(
    OUTPUT_DIR / "reference_measurements.csv", index=False, encoding="utf-8-sig"
)
reference_table.head()


,sample_name,image_id,image_path,ca_deg,hva_deg,nmp_percent,si_percent,csi_percent
0,CD_B1_000004_L_S00,9,CD/batch1/CD_B1_000004_L_S00.png,16.101488,22.718162,18.556561,105.995817,58.300790
1,CD_B1_000004_R_S00,10,CD/batch1/CD_B1_000004_R_S00.png,37.874116,16.522149,16.768069,65.822541,37.981912
2,CD_B1_000007_L_S00,15,CD/batch1/CD_B1_000007_L_S00.png,12.706453,8.021600,19.992562,111.104952,57.394805
3,CD_B1_000007_R_S00,16,CD/batch1/CD_B1_000007_R_S00.png,4.619122,19.364715,23.265258,135.391933,74.660039
4,CD_B1_000009_L_S00,21,CD/batch1/CD_B1_000009_L_S00.png,19.996570,9.262326,14.705216,99.606294,56.457205


In [4]:
models = [
    "simplebaseline_res50", "hrnet_w32", "litehrnet18",
    "rtmpose_m", "vitpose_b", "mspn_4stage",
]

prediction_tables = {}
for model in models:
    with Path(PREDICTION_PATH.format(model=model)).open("rb") as f:
        predictions = pickle.load(f)

    points = {
        sample["img_id"]: sample["pred_instances"]["keypoints"]
        for sample in predictions
    }
    table = make_table(points)
    table.to_csv(
        OUTPUT_DIR / f"{model}_measurements.csv", index=False, encoding="utf-8-sig"
    )
    prediction_tables[model] = table
    print(f"{model}: saved {len(table)} rows")

print("Output directory:", OUTPUT_DIR)


simplebaseline_res50: saved 1397 rows
hrnet_w32: saved 1397 rows
litehrnet18: saved 1397 rows
rtmpose_m: saved 1397 rows
vitpose_b: saved 1397 rows
mspn_4stage: saved 1397 rows
Output directory: ..\..\test_results\0_measurements
